# ZAALIMA Project-02 — Week 2: Context-Aware Two-Tower Recommendation Model

This is a clean, end-to-end Google Colab notebook for the Week 2 requirements in the project brief.

**Week 2 deliverables**
1. Build a Query Tower for user context and a Candidate Tower for article context.
2. Train the two-tower retrieval model using in-batch negative sampling.
3. Evaluate recommendation quality using Recall@K and NDCG@K.

**Dataset:** H&M Personalized Fashion Recommendations (Kaggle), using `articles.csv`, `customers.csv`, and `transactions_train.csv`.

**Colab storage:** this notebook expects the dataset files in `/content/drive/MyDrive/hm_project/`. The interaction data is sampled to 25,000 rows for a manageable Colab experiment. The article catalogue and customer reference data are loaded in full. Metrics must be described as results on this sampled experiment, not the full transaction dataset.

## 1. Connect Google Drive and verify the H&M dataset

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
DATA_PATH = "/content/drive/MyDrive/hm_project"
required_files = [
    "articles.csv",
    "customers.csv",
    "transactions_train.csv"
]
missing = [
    name for name in required_files
    if not os.path.exists(os.path.join(DATA_PATH, name))
]
if missing:
    raise FileNotFoundError(
        f"Missing files in {DATA_PATH}: {missing}. "
        "Place the H&M Kaggle CSV files in this Drive folder."
    )

print("Dataset folder:", DATA_PATH)
for name in required_files:
    size_mb = os.path.getsize(os.path.join(DATA_PATH, name)) / (1024 ** 2)
    print(f"{name}: {size_mb:.1f} MB")

## 2. Import libraries and load reference tables

In [ ]:
import gc
import json
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from pathlib import Path

print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

articles = pd.read_csv(
    os.path.join(DATA_PATH, "articles.csv"),
    dtype={"article_id": str}
)
customers = pd.read_csv(
    os.path.join(DATA_PATH, "customers.csv"),
    dtype={"customer_id": str}
)

assert articles["article_id"].is_unique
assert customers["customer_id"].is_unique

print("Articles:", len(articles))
print("Customers:", len(customers))
print("Reference tables loaded and validated.")

## 3. Prepare a manageable interaction sample

In [ ]:
# Read five 100k-row chunks and sample 5k interactions from each.
# This keeps the Colab experiment manageable while preserving real H&M interactions.
transaction_path = os.path.join(DATA_PATH, "transactions_train.csv")
sampled_chunks = []

reader = pd.read_csv(
    transaction_path,
    usecols=["customer_id", "article_id"],
    dtype={"customer_id": str, "article_id": str},
    chunksize=100_000
)

for chunk_number, chunk in enumerate(reader):
    sampled_chunks.append(
        chunk.sample(
            n=min(5_000, len(chunk)),
            random_state=42 + chunk_number
        )
    )
    if chunk_number == 4:
        break

interactions = pd.concat(sampled_chunks, ignore_index=True)
interactions = interactions.drop_duplicates(
    subset=["customer_id", "article_id"]
).reset_index(drop=True)

print("Sampled unique interactions:", len(interactions))
display(interactions.head())

## 4. Join customer and article context features

In [ ]:
customer_columns = [
    "customer_id",
    "club_member_status",
    "fashion_news_frequency",
    "age"
]
metadata_columns = [
    "product_type_name",
    "product_group_name",
    "colour_group_name",
    "department_name",
    "garment_group_name"
]

customer_features = customers[customer_columns].copy()
customer_features["customer_id"] = customer_features["customer_id"].astype(str)
customer_features["club_member_status"] = (
    customer_features["club_member_status"].fillna("Unknown").astype(str)
)
customer_features["fashion_news_frequency"] = (
    customer_features["fashion_news_frequency"].fillna("Unknown").astype(str)
)
customer_features["age"] = pd.to_numeric(
    customer_features["age"], errors="coerce"
).fillna(0).astype("float32")

article_data = articles[["article_id"] + metadata_columns].copy()
article_data["article_id"] = article_data["article_id"].astype(str)
for column in metadata_columns:
    article_data[column] = article_data[column].fillna("Unknown").astype(str)

data = interactions.merge(
    customer_features,
    on="customer_id",
    how="inner",
    validate="many_to_one"
)
data = data.merge(
    article_data,
    on="article_id",
    how="inner",
    validate="many_to_one"
)
data = data.drop_duplicates(
    subset=["customer_id", "article_id"]
).reset_index(drop=True)

print("Prepared interaction rows:", len(data))
print("Missing values:", int(data.isna().sum().sum()))
display(data.head())

## 5. Create train/validation split and vocabularies

In [ ]:
train_df, val_df = train_test_split(
    data,
    test_size=0.10,
    random_state=42
)
train_df = train_df.reset_index(drop=True).copy()
val_df = val_df.reset_index(drop=True).copy()

# Full reference vocabularies ensure the model can represent the full catalogue.
customer_vocab = customers["customer_id"].astype(str).unique().tolist()
club_vocab = (
    customers["club_member_status"].fillna("Unknown").astype(str).unique().tolist()
)
news_vocab = (
    customers["fashion_news_frequency"].fillna("Unknown").astype(str).unique().tolist()
)
article_vocab = article_data["article_id"].astype(str).unique().tolist()

metadata_vocabs = {
    col: article_data[col].fillna("Unknown").astype(str).unique().tolist()
    for col in metadata_columns
}

print("Training interactions:", len(train_df))
print("Validation interactions:", len(val_df))
print("Customer vocabulary:", len(customer_vocab))
print("Article vocabulary:", len(article_vocab))
for col, vocab in metadata_vocabs.items():
    print(f"{col} vocabulary:", len(vocab))

## 6. Build the Query Tower, Candidate Tower and Two-Tower model

In [ ]:
class QueryTower(tf.keras.Model):
    def __init__(
        self, customer_vocab, club_vocab, news_vocab,
        embedding_dim=32
    ):
        super().__init__()
        self.customer_lookup = tf.keras.layers.StringLookup(
            vocabulary=customer_vocab, mask_token=None
        )
        self.club_lookup = tf.keras.layers.StringLookup(
            vocabulary=club_vocab, mask_token=None
        )
        self.news_lookup = tf.keras.layers.StringLookup(
            vocabulary=news_vocab, mask_token=None
        )

        self.customer_embedding = tf.keras.layers.Embedding(
            self.customer_lookup.vocabulary_size(), embedding_dim
        )
        self.club_embedding = tf.keras.layers.Embedding(
            self.club_lookup.vocabulary_size(), 8
        )
        self.news_embedding = tf.keras.layers.Embedding(
            self.news_lookup.vocabulary_size(), 8
        )

        self.dense1 = tf.keras.layers.Dense(64, activation="relu")
        self.dense2 = tf.keras.layers.Dense(embedding_dim)

    def call(self, inputs, training=False):
        customer_id = self.customer_lookup(inputs["customer_id"])
        club_status = self.club_lookup(inputs["club_status"])
        news_frequency = self.news_lookup(inputs["news_frequency"])

        customer_vector = self.customer_embedding(customer_id)
        club_vector = self.club_embedding(club_status)
        news_vector = self.news_embedding(news_frequency)

        age = tf.cast(inputs["age"], tf.float32)
        age = tf.expand_dims(age, axis=-1) / 100.0

        combined = tf.concat(
            [customer_vector, club_vector, news_vector, age],
            axis=1
        )
        x = self.dense1(combined)
        return self.dense2(x)


class CandidateTower(tf.keras.Model):
    def __init__(self, article_vocab, metadata_vocabs, embedding_dim=32):
        super().__init__()
        self.article_lookup = tf.keras.layers.StringLookup(
            vocabulary=article_vocab, mask_token=None
        )
        self.article_embedding = tf.keras.layers.Embedding(
            self.article_lookup.vocabulary_size(), embedding_dim
        )

        self.metadata_lookups = {}
        self.metadata_embeddings = {}
        for feature, vocab in metadata_vocabs.items():
            self.metadata_lookups[feature] = tf.keras.layers.StringLookup(
                vocabulary=vocab, mask_token=None
            )
            self.metadata_embeddings[feature] = tf.keras.layers.Embedding(
                self.metadata_lookups[feature].vocabulary_size(), 8
            )

        self.dense1 = tf.keras.layers.Dense(64, activation="relu")
        self.dropout = tf.keras.layers.Dropout(0.2)
        self.dense2 = tf.keras.layers.Dense(embedding_dim)

    def call(self, inputs, training=False):
        article_index = self.article_lookup(inputs["article_id"])
        vectors = [self.article_embedding(article_index)]

        for feature in self.metadata_lookups:
            index = self.metadata_lookups[feature](inputs[feature])
            vectors.append(self.metadata_embeddings[feature](index))

        combined = tf.concat(vectors, axis=1)
        x = self.dense1(combined)
        x = self.dropout(x, training=training)
        return self.dense2(x)


class TwoTowerModel(tf.keras.Model):
    def __init__(self, query_tower, candidate_tower):
        super().__init__()
        self.query_tower = query_tower
        self.candidate_tower = candidate_tower

    def call(self, inputs, training=False):
        user_embeddings = self.query_tower(
            inputs["customer"], training=training
        )
        item_embeddings = self.candidate_tower(
            inputs["candidate"], training=training
        )
        return tf.matmul(
            user_embeddings, item_embeddings, transpose_b=True
        )


EMBEDDING_DIM = 32
query_tower = QueryTower(
    customer_vocab, club_vocab, news_vocab, EMBEDDING_DIM
)
candidate_tower = CandidateTower(
    article_vocab, metadata_vocabs, EMBEDDING_DIM
)
model = TwoTowerModel(query_tower, candidate_tower)

print("Query Tower, Candidate Tower and Two-Tower model created.")

## 7. Create TensorFlow datasets and test the forward pass

In [ ]:
def make_tf_dataset(frame, batch_size=256, shuffle=False):
    customer_inputs = {
        "customer_id": frame["customer_id"].astype(str).values,
        "club_status": frame["club_member_status"].astype(str).values,
        "news_frequency": frame["fashion_news_frequency"].astype(str).values,
        "age": frame["age"].astype("float32").values
    }
    candidate_inputs = {
        "article_id": frame["article_id"].astype(str).values
    }
    for feature in metadata_columns:
        candidate_inputs[feature] = frame[feature].astype(str).values

    dataset = tf.data.Dataset.from_tensor_slices({
        "customer": customer_inputs,
        "candidate": candidate_inputs
    })
    if shuffle:
        dataset = dataset.shuffle(
            buffer_size=min(len(frame), 10_000),
            reshuffle_each_iteration=True
        )
    return dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)


BATCH_SIZE = 256
train_ds = make_tf_dataset(train_df, BATCH_SIZE, shuffle=True)
val_ds = make_tf_dataset(val_df, BATCH_SIZE, shuffle=False)

sample_batch = next(iter(train_ds))
sample_scores = model(sample_batch, training=False)

print("Similarity matrix shape:", sample_scores.shape)
print("Forward pass successful:",
      bool(tf.reduce_all(tf.math.is_finite(sample_scores)).numpy()))

## 8. Define in-batch negative-sampling loss

For every mini-batch, the distinct articles in that batch form the candidate pool. For a customer, articles they interacted with in the same batch are positives; other customers' articles are sampled negatives. The multi-positive objective prevents another positive for the same customer from being incorrectly treated as a negative.

In [ ]:
def multi_positive_loss(batch, temperature=0.1, training=False):
    customer_ids = batch["customer"]["customer_id"]
    article_ids = batch["candidate"]["article_id"]

    user_embeddings = model.query_tower(
        batch["customer"], training=training
    )
    item_embeddings = model.candidate_tower(
        batch["candidate"], training=training
    )

    unique_articles, article_indices = tf.unique(article_ids)
    first_indices = tf.math.unsorted_segment_min(
        tf.range(tf.shape(article_ids)[0]),
        article_indices,
        tf.shape(unique_articles)[0]
    )
    unique_item_embeddings = tf.gather(item_embeddings, first_indices)

    scores = tf.matmul(
        user_embeddings, unique_item_embeddings, transpose_b=True
    ) / temperature

    same_customer = tf.equal(
        customer_ids[:, None], customer_ids[None, :]
    )
    same_article = tf.equal(
        article_ids[:, None], unique_articles[None, :]
    )
    positive_mask = tf.reduce_any(
        same_customer[:, :, None] & same_article[None, :, :],
        axis=1
    )

    tf.debugging.assert_positive(
        tf.reduce_sum(tf.cast(positive_mask, tf.int32), axis=1),
        message="Each query must have at least one positive candidate."
    )

    positive_scores = tf.where(
        positive_mask,
        scores,
        tf.constant(-1e9, dtype=scores.dtype)
    )
    per_query_loss = (
        tf.reduce_logsumexp(scores, axis=1)
        - tf.reduce_logsumexp(positive_scores, axis=1)
    )
    return tf.reduce_mean(per_query_loss)


audit_customer_ids = sample_batch["customer"]["customer_id"]
audit_article_ids = sample_batch["candidate"]["article_id"]
unique_batch_articles, _ = tf.unique(audit_article_ids)
same_customer = tf.equal(
    audit_customer_ids[:, None], audit_customer_ids[None, :]
)
same_article = tf.equal(
    audit_article_ids[:, None], unique_batch_articles[None, :]
)
positive_mask = tf.reduce_any(
    same_customer[:, :, None] & same_article[None, :, :], axis=1
)
positive_counts = tf.reduce_sum(tf.cast(positive_mask, tf.int32), axis=1)

print("Distinct batch candidates:", int(tf.shape(unique_batch_articles)[0]))
print("Average positives/query:",
      float(tf.reduce_mean(tf.cast(positive_counts, tf.float32)).numpy()))
print("Average sampled negatives/query:",
      float(tf.reduce_mean(
          tf.cast(tf.shape(unique_batch_articles)[0] - positive_counts, tf.float32)
      ).numpy()))
print("Negative-sampling audit passed.")

## 9. Train the model and monitor validation loss

In [ ]:
optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001)

def train_step(batch):
    with tf.GradientTape() as tape:
        loss = multi_positive_loss(batch, training=True)
    gradients = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(
        (grad, var)
        for grad, var in zip(gradients, model.trainable_variables)
        if grad is not None
    )
    return loss

def validation_step(batch):
    return multi_positive_loss(batch, training=False)


EPOCHS = 3
history = {"train_loss": [], "val_loss": []}

for epoch in range(EPOCHS):
    train_losses = []
    for batch in train_ds:
        loss = train_step(batch)
        train_losses.append(float(loss.numpy()))

    val_losses = [
        float(validation_step(batch).numpy())
        for batch in val_ds
    ]

    train_loss = float(np.mean(train_losses))
    val_loss = float(np.mean(val_losses))
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"Training Loss: {train_loss:.4f} | "
        f"Validation Loss: {val_loss:.4f}"
    )

print("Training completed.")

## 10. Plot training and validation loss

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(
    range(1, len(history["train_loss"]) + 1),
    history["train_loss"], marker="o", label="Training Loss"
)
plt.plot(
    range(1, len(history["val_loss"]) + 1),
    history["val_loss"], marker="o", label="Validation Loss"
)
plt.xlabel("Epoch")
plt.ylabel("Multi-positive contrastive loss")
plt.title("Two-Tower Training History")
plt.xticks(range(1, len(history["train_loss"]) + 1))
plt.grid(True)
plt.legend()
plt.show()

## 11. Prepare full-catalogue evaluation

Each validation customer is evaluated against every article in the H&M article catalogue. Articles that customer already interacted with in the training split are excluded. Only validation articles not seen in that customer's training history count as relevant targets.

In [ ]:
eval_users = (
    val_df[customer_columns]
    .drop_duplicates(subset=["customer_id"])
    .reset_index(drop=True)
    .copy()
)
eval_users["customer_id"] = eval_users["customer_id"].astype(str)

catalog = article_data[["article_id"] + metadata_columns].copy()
catalog = catalog.drop_duplicates("article_id").reset_index(drop=True)
catalog["article_id"] = catalog["article_id"].astype(str)
catalog_ids = catalog["article_id"].tolist()
catalog_id_to_index = {
    article_id: i for i, article_id in enumerate(catalog_ids)
}

train_seen = (
    train_df.groupby("customer_id")["article_id"]
    .apply(lambda values: set(values.astype(str)))
    .to_dict()
)
ground_truth_all = (
    val_df.groupby("customer_id")["article_id"]
    .apply(lambda values: set(values.astype(str)))
    .to_dict()
)
ground_truth = {
    str(customer_id): items - train_seen.get(customer_id, set())
    for customer_id, items in ground_truth_all.items()
}
ground_truth = {
    customer_id: items for customer_id, items in ground_truth.items()
    if items
}

unknown_targets = {
    item for items in ground_truth.values() for item in items
    if item not in catalog_id_to_index
}
if unknown_targets:
    raise ValueError(
        f"{len(unknown_targets)} validation targets are missing from catalogue."
    )

print("Validation customers:", len(eval_users))
print("Customers with unseen validation targets:", len(ground_truth))
print("Full article catalogue:", len(catalog_ids))

## 12. Generate embeddings for the full article catalogue

In [ ]:
def candidate_input_dict(frame):
    inputs = {
        "article_id": tf.constant(frame["article_id"].astype(str).values)
    }
    for feature in metadata_columns:
        inputs[feature] = tf.constant(
            frame[feature].fillna("Unknown").astype(str).values
        )
    return inputs

catalog_embedding_parts = []
CATALOG_BATCH_SIZE = 4096

for start in range(0, len(catalog), CATALOG_BATCH_SIZE):
    part = catalog.iloc[start:start + CATALOG_BATCH_SIZE]
    part_embeddings = candidate_tower(
        candidate_input_dict(part), training=False
    )
    catalog_embedding_parts.append(part_embeddings)

catalog_embeddings = tf.concat(catalog_embedding_parts, axis=0)
print("Full-catalogue embedding matrix:", catalog_embeddings.shape)
print("Embeddings finite:",
      bool(tf.reduce_all(tf.math.is_finite(catalog_embeddings)).numpy()))

## 13. Evaluate Recall@5, Recall@10, NDCG@5 and NDCG@10

In [ ]:
def dcg_at_k(relevance, k):
    relevance = np.asarray(relevance, dtype=np.float32)[:k]
    if len(relevance) == 0:
        return 0.0
    discounts = np.log2(np.arange(2, len(relevance) + 2))
    return float(np.sum(relevance / discounts))


def evaluate_recommender(users, user_batch_size=128, k_values=(5, 10)):
    max_k = max(k_values)
    results = {f"Recall@{k}": [] for k in k_values}
    results.update({f"NDCG@{k}": [] for k in k_values})

    for start in range(0, len(users), user_batch_size):
        batch_users = users.iloc[start:start + user_batch_size]

        query_inputs = {
            "customer_id": tf.constant(batch_users["customer_id"].astype(str).values),
            "club_status": tf.constant(batch_users["club_member_status"].fillna("Unknown").astype(str).values),
            "news_frequency": tf.constant(batch_users["fashion_news_frequency"].fillna("Unknown").astype(str).values),
            "age": tf.constant(pd.to_numeric(
                batch_users["age"], errors="coerce"
            ).fillna(0).astype("float32").values)
        }
        query_embeddings = query_tower(query_inputs, training=False)

        scores = tf.matmul(
            query_embeddings, catalog_embeddings, transpose_b=True
        ).numpy()

        # Remove training-history items from each customer's candidate list.
        customer_ids = batch_users["customer_id"].astype(str).tolist()
        for row, customer_id in enumerate(customer_ids):
            seen_indices = [
                catalog_id_to_index[item]
                for item in train_seen.get(customer_id, set())
                if item in catalog_id_to_index
            ]
            if seen_indices:
                scores[row, seen_indices] = -np.inf

        # Retrieve the top max_k items for every user.
        top_indices = np.argpartition(
            scores, -max_k, axis=1
        )[:, -max_k:]
        top_scores = np.take_along_axis(scores, top_indices, axis=1)
        order = np.argsort(-top_scores, axis=1)
        top_indices = np.take_along_axis(top_indices, order, axis=1)

        for row, customer_id in enumerate(customer_ids):
            relevant = ground_truth.get(customer_id, set())
            if not relevant:
                continue

            recommended = [catalog_ids[index] for index in top_indices[row]]
            relevance = np.array(
                [1.0 if item in relevant else 0.0 for item in recommended],
                dtype=np.float32
            )

            for k in k_values:
                recall = float(relevance[:k].sum()) / len(relevant)
                dcg = dcg_at_k(relevance, k)
                ideal_count = min(len(relevant), k)
                idcg = dcg_at_k(np.ones(ideal_count), k)
                ndcg = dcg / idcg if idcg > 0 else 0.0

                results[f"Recall@{k}"].append(recall)
                results[f"NDCG@{k}"].append(ndcg)

    return {
        metric: float(np.mean(values)) if values else 0.0
        for metric, values in results.items()
    }


if len(catalog_ids) < 10:
    raise ValueError("At least 10 catalogue articles are required for evaluation.")

week2_metrics = evaluate_recommender(
    eval_users,
    user_batch_size=128,
    k_values=(5, 10)
)

print("FULL-CATALOGUE WEEK 2 METRICS")
for name, value in week2_metrics.items():
    print(f"{name}: {value:.4f}")

## 14. Display and validate evaluation results

In [ ]:
metrics_df = pd.DataFrame([
    {"Metric": name, "Score": score}
    for name, score in week2_metrics.items()
])
display(metrics_df)

assert set(week2_metrics) == {
    "Recall@5", "Recall@10", "NDCG@5", "NDCG@10"
}
assert all(
    np.isfinite(score) and 0.0 <= score <= 1.0
    for score in week2_metrics.values()
)
print("All evaluation metrics are valid values between 0 and 1.")

## 15. Save the Week 2 model and report to the GitHub project folder

In [ ]:
PROJECT_DIR = Path("/content/hm-recommendation-engine")

# Clone the team's Week 2 branch if this is a fresh Colab runtime.
import subprocess
if not (PROJECT_DIR / ".git").exists():
    if PROJECT_DIR.exists():
        import shutil
        shutil.rmtree(PROJECT_DIR)
    subprocess.run(
        [
            "git", "clone", "-b", "week-2/deep-learning-model",
            "https://github.com/afnakurikkal/hm-recommendation-engine.git",
            str(PROJECT_DIR)
        ],
        check=True
    )

ARTIFACT_DIR = PROJECT_DIR / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

weights_path = ARTIFACT_DIR / "two_tower_week2.weights.h5"
model.save_weights(str(weights_path))

metrics_path = ARTIFACT_DIR / "week2_evaluation_metrics.json"
with open(metrics_path, "w", encoding="utf-8") as file:
    json.dump({
        "project": "ZAALIMA Project-02",
        "model": "Context-Aware Two-Tower Neural Recommendation Engine",
        "dataset": "H&M Personalized Fashion Recommendations (Kaggle)",
        "interaction_sample_size": int(len(train_df) + len(val_df)),
        "training_interactions": int(len(train_df)),
        "validation_interactions": int(len(val_df)),
        "catalogue_articles": int(len(catalog)),
        "embedding_dimension": EMBEDDING_DIM,
        "training_epochs": EPOCHS,
        "negative_sampling": "In-batch negatives with multi-positive contrastive loss",
        "evaluation": "Full article catalogue; training-seen items excluded per customer",
        "metrics": week2_metrics
    }, file, indent=4)

report_lines = [
    "# Week 2 — Deep Learning Model Architecture",
    "",
    "## Implemented",
    "- Context-aware Query Tower and metadata-aware Candidate Tower",
    "- Two-Tower user/item embedding model",
    "- In-batch negative sampling with multi-positive contrastive loss",
    "- Model training and validation",
    "- Full-catalogue Recall@K and NDCG@K evaluation",
    "",
    "## Dataset",
    f"- Training interactions: {len(train_df):,}",
    f"- Validation interactions: {len(val_df):,}",
    f"- Catalogue articles: {len(catalog):,}",
    "- Interactions sampled to 25,000 rows for this Colab experiment.",
    "",
    "## Evaluation"
]
report_lines.extend([
    f"- {name}: {score:.4f}"
    for name, score in week2_metrics.items()
])
report_lines.extend([
    "",
    "## Scope note",
    "Metrics describe the sampled experiment, not a full-transaction production run.",
    "Raw customer data and source CSV files are not included in the artifacts."
])
(ARTIFACT_DIR / "WEEK2_REPORT.md").write_text(
    "\n".join(report_lines), encoding="utf-8"
)

print("Saved model:", weights_path)
print("Saved metrics:", metrics_path)
print("Saved report:", ARTIFACT_DIR / "WEEK2_REPORT.md")

## 16. Week 2 completion checklist

After every cell runs successfully:

- [x] Query Tower built for user context
- [x] Candidate Tower built for item context
- [x] User and item embeddings generated
- [x] Two-Tower similarity scoring implemented
- [x] In-batch negative sampling implemented
- [x] Model trained and validation loss monitored
- [x] Recall@5 and Recall@10 calculated
- [x] NDCG@5 and NDCG@10 calculated
- [x] Model weights, metrics and report saved

**Week 3 is not included:** model/embedding export for serving, Redis feature store, and ANN retrieval are Week 3 tasks.